# CLIP smoke check on Apple MPS

Run this on the Mac before CuPL or Stanford experiments. It checks a small MPS gradient and the CLIP text encoder. It does not train or validate paper accuracy. If MPS is unavailable, the notebook stops with an explicit error.


In [ ]:
import torch
from transformers import CLIPModel, CLIPTokenizer
EXPECTED_DEVICE = "mps"
assert getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available(), (
    "Apple MPS is unavailable. Check macOS, the PyTorch build, and the Apple GPU."
)
device = torch.device(EXPECTED_DEVICE)
print("PyTorch:", torch.__version__, "device:", device)
x = torch.ones((8, 8), device=device, requires_grad=True)
(x @ x).sum().backward()
torch.mps.synchronize()
assert x.grad is not None and torch.isfinite(x.grad).all()
print("MPS matrix forward/backward: OK")


In [ ]:
model_id = "openai/clip-vit-base-patch32"
tokenizer = CLIPTokenizer.from_pretrained(model_id)
model = CLIPModel.from_pretrained(model_id).to(device).eval()
inputs = tokenizer(["a photo of a Birman cat", "a photo of a Ragdoll cat"],
                   padding=True, truncation=True, return_tensors="pt").to(device)
with torch.no_grad():
    features = model.get_text_features(**inputs)
torch.mps.synchronize()
assert features.shape[0] == 2 and torch.isfinite(features).all()
print("CLIP text encoder on MPS: OK; features:", tuple(features.shape))
